# Event Bus and Messaging — Pub/Sub, Request/Reply, and Actor Patterns

**Multigen SDK — Notebook 17**

---

This notebook covers the **InMemoryBus** — the Multigen SDK's message-passing infrastructure for connecting agents in complex topologies.

While `Chain`, `Parallel`, and `Graph` handle **structured** control flow, sometimes agents need to communicate in more **flexible, decoupled** ways:
- An agent that discovers new tasks wants to broadcast them without knowing who cares
- Multiple specialist agents want to subscribe to a shared topic and each process messages independently
- A coordinating agent wants to request work from a pool of workers without caring which one responds

These patterns come from **distributed systems messaging** — and the `InMemoryBus` brings them to in-process agentic workflows.

---

### Messaging Patterns Covered

| Pattern | Description | SDK Method |
|---|---|---|
| **Pub/Sub** | Publisher sends, multiple subscribers receive | `@bus.subscribe`, `bus.publish` |
| **Request/Reply** | Caller sends, handler responds with correlation | `@bus.on_request`, `bus.request` |
| **Broadcast** | Message reaches ALL agents | `bus.broadcast` |
| **Push/Pull** | Work queue with round-robin distribution | `bus.add_worker`, `bus.push` |
| **Direct Queue** | Point-to-point actor messaging | `bus.send`, `bus.receive` |

## Section 1: Messaging Patterns Overview

Before diving into code, let's understand *why* agents need these patterns.

### The Limits of Direct Calls

In a `Chain`, Agent A directly calls Agent B:
```
A.run() → B.run() → C.run()
```

This is simple and works well for linear workflows. But it creates **tight coupling**:
- A must know B exists
- A must know B's interface
- If B fails, A is blocked
- Adding a 4th agent D requires modifying the chain definition

### Message-Passing Solves This

With a message bus, agents communicate **through topics** rather than direct references:
```
A publishes → "analysis.done" topic → [B, C, D all subscribed]
```

Now:
- A doesn't know or care how many subscribers exist
- Adding agent D just means D subscribes to the topic
- Removing B doesn't affect A or C
- This is **decoupled** fan-out behavior

### ASCII Diagram: The Five Patterns

```
PUB/SUB:                  REQUEST/REPLY:
Publisher ──► [topic] ──► Sub A     Caller ──► [topic] ──► Handler
                     ├──► Sub B        └─────────────────── response (same request)
                     └──► Sub C

BROADCAST:                PUSH/PULL:
Sender ──► [ALL agents]   Producer ──► [queue] ──► Worker A
                                              ├──► Worker B  (round-robin)
                                              └──► Worker C

DIRECT QUEUE:
Sender ──► [agent_id queue] ──► Target agent (reads from its own inbox)
```

## Section 2: The InMemoryBus — Creating an Instance

The `InMemoryBus` is a fully in-process message bus. All messaging is done through asyncio queues — no network, no serialization overhead.

There are two ways to get a bus:
1. **`InMemoryBus()`** — Create a fresh, isolated bus instance
2. **`get_default_bus()`** — Get the process-level singleton bus (shared across all code)

In [ ]:
# SDK Imports
import asyncio
import time
import uuid
from multigen import InMemoryBus, Message, get_default_bus, FunctionAgent

print("Imports loaded successfully.")

# --- Method 1: Create a fresh bus ---
bus = InMemoryBus()
print(f"\nFresh bus created: {bus}")
print(f"Type: {type(bus).__name__}")

# --- Method 2: Get the process-level singleton ---
# All code in the same process that calls get_default_bus() gets the SAME bus instance
default_bus = get_default_bus()
print(f"\nDefault (singleton) bus: {default_bus}")
print(f"Are they the same instance? {bus is default_bus}")
# False — bus is a fresh instance, default_bus is the global singleton

# Multiple calls to get_default_bus() return the same object
default_bus_again = get_default_bus()
print(f"Two calls to get_default_bus() same object? {default_bus is default_bus_again}")
# True — it's a singleton

print()
print("Use InMemoryBus() for isolated unit tests or scoped workflows.")
print("Use get_default_bus() when you want agents across the codebase to share one bus.")

## Section 3: The Message Dataclass

Every message flowing through the bus is a `Message` instance. Let's look at all its fields.

```python
@dataclass
class Message:
    topic: str              # Routing key — subscribers filter by topic
    content: Any            # The actual payload — any Python object
    id: str                 # Auto-generated UUID for the message
    sender: str             # Identifier of who sent the message
    reply_to: str | None    # For request/reply: topic to send the response to
    ttl: float | None       # Time-to-live in seconds; None = no expiry
    priority: int           # Higher = processed first (default 0)
    headers: dict           # Optional metadata key-value pairs
    timestamp: float        # Unix timestamp when message was created
```

In [ ]:
# --- Explore the Message dataclass in detail ---

# Minimal message — only topic and content are required
minimal_msg = Message(
    topic="analysis.result",
    content={"score": 0.87, "label": "positive"}
)

print("=" * 60)
print("MINIMAL MESSAGE")
print("=" * 60)
print(f"id:        {minimal_msg.id}        ← auto-generated UUID")
print(f"topic:     {minimal_msg.topic}")
print(f"content:   {minimal_msg.content}")
print(f"sender:    {minimal_msg.sender!r}   ← defaults to None")
print(f"reply_to:  {minimal_msg.reply_to!r}   ← None = no reply expected")
print(f"ttl:       {minimal_msg.ttl!r}         ← None = lives forever")
print(f"priority:  {minimal_msg.priority}              ← 0 = normal priority")
print(f"headers:   {minimal_msg.headers}           ← empty dict")
print(f"timestamp: {minimal_msg.timestamp:.3f}   ← Unix epoch")

print()

# Full message with all fields specified
full_msg = Message(
    topic="tasks.new",
    content="Analyze the quarterly report",
    sender="CoordinatorAgent",
    reply_to="tasks.completed",   # Response should go to this topic
    ttl=30.0,                     # Expires in 30 seconds if not delivered
    priority=10,                  # High priority — process before normal messages
    headers={
        "correlation_id": str(uuid.uuid4()),
        "source_workflow": "q4-analysis-pipeline",
        "retry_count": 0
    }
)

print("=" * 60)
print("FULL MESSAGE WITH ALL FIELDS")
print("=" * 60)
print(f"id:         {full_msg.id}")
print(f"topic:      {full_msg.topic}")
print(f"content:    {full_msg.content!r}")
print(f"sender:     {full_msg.sender}")
print(f"reply_to:   {full_msg.reply_to}")
print(f"ttl:        {full_msg.ttl}s")
print(f"priority:   {full_msg.priority}  ← higher = processed first")
print(f"headers:    {full_msg.headers}")

print()
print("Message ID is unique per message — useful for deduplication and tracing.")
print("Two messages with the same content will have different IDs.")
msg_a = Message(topic="test", content="hello")
msg_b = Message(topic="test", content="hello")
print(f"msg_a.id: {msg_a.id}")
print(f"msg_b.id: {msg_b.id}")
print(f"Are IDs different? {msg_a.id != msg_b.id}")

## Section 4: Pattern 1 — Pub/Sub

**Publisher/Subscriber** is the most common messaging pattern:
- **Publisher** sends messages to a topic without knowing who's listening
- **Subscriber(s)** register interest in a topic and receive all messages sent to it

The `@bus.subscribe("topic")` decorator registers a coroutine as a subscriber. Multiple subscribers on the same topic each receive a **copy** of every message (fan-out).

Use cases:
- Event notifications (agent A publishes results, agents B and C both want to log them)
- Multi-consumer workflows (a search result is fed to multiple analysis agents)
- Monitoring/observability (subscribe to all topics to log everything)

In [ ]:
# --- Pattern 1: Pub/Sub ---

# Create an isolated bus for this demo
pubsub_bus = InMemoryBus()

# Collect received messages for inspection
received_messages = []

# --- Register subscribers ---
# @pubsub_bus.subscribe("topic") registers a coroutine as a message handler
# The handler receives the full Message object

@pubsub_bus.subscribe("news.articles")
async def summarizer_subscriber(msg: Message):
    """
    Subscriber 1: Summarizes incoming articles.
    Registered on topic 'news.articles'.
    """
    summary = f"SUMMARY of '{msg.content}' by summarizer"
    received_messages.append({"subscriber": "summarizer", "content": msg.content, "result": summary})
    print(f"  [Summarizer] Received: {msg.content!r} → {summary}")

@pubsub_bus.subscribe("news.articles")
async def sentiment_subscriber(msg: Message):
    """
    Subscriber 2: Analyzes sentiment of incoming articles.
    Also registered on topic 'news.articles' — will receive ALL messages on this topic.
    Fan-out: both summarizer and sentiment_subscriber get each message.
    """
    import random
    sentiment = random.choice(["positive", "neutral", "negative"])
    received_messages.append({"subscriber": "sentiment", "content": msg.content, "result": sentiment})
    print(f"  [Sentiment]  Received: {msg.content!r} → {sentiment}")

@pubsub_bus.subscribe("news.articles")
async def keyword_subscriber(msg: Message):
    """
    Subscriber 3: Extracts keywords.
    Three subscribers on the same topic = all three fire for each published message.
    """
    words = msg.content.split()[:3] if isinstance(msg.content, str) else []
    received_messages.append({"subscriber": "keywords", "content": msg.content, "result": words})
    print(f"  [Keywords]   Received: {msg.content!r} → keywords: {words}")

print("Three subscribers registered on topic 'news.articles':")
print("  - summarizer_subscriber")
print("  - sentiment_subscriber")
print("  - keyword_subscriber")
print()

# --- Publish messages ---
async def demo_pubsub():
    articles = [
        "Tech companies report strong Q4 earnings despite market volatility",
        "New AI regulation framework proposed by European Parliament",
        "Climate summit reaches landmark agreement on emissions"
    ]
    
    print("Publishing 3 articles to 'news.articles' topic:")
    print("Each article will be received by ALL 3 subscribers (fan-out)")
    print("-" * 60)
    
    for i, article in enumerate(articles, 1):
        print(f"\nPublishing article {i}: '{article[:50]}...'")
        
        # Create and publish a message
        msg = Message(
            topic="news.articles",
            content=article,
            sender="NewsIngestionAgent",
            headers={"article_id": f"art-{i:04d}", "source": "reuters"}
        )
        await pubsub_bus.publish(msg)
        # Small delay to let subscribers process
        await asyncio.sleep(0.01)
    
    # Give subscribers time to finish
    await asyncio.sleep(0.05)

asyncio.run(demo_pubsub())

print()
print("=" * 60)
print(f"Total messages received across all subscribers: {len(received_messages)}")
print(f"Expected: 3 articles × 3 subscribers = {3*3} messages")
print()
print("Each article was delivered to ALL 3 subscribers (fan-out behavior).")

## Section 5: Multiple Subscribers — Fan-Out Behavior

The previous demo showed fan-out in action. Let's explore it more deeply.

**Fan-out** means one published message creates N deliveries — one per subscriber.

Key properties:
- Each subscriber gets an **independent copy** of the message
- Subscribers run **concurrently** (all start processing before any one finishes)
- If one subscriber fails, others still receive the message
- Subscribers can be on different parts of the codebase — they don't know about each other

In [ ]:
# --- Fan-out timing analysis ---

fanout_bus = InMemoryBus()
fanout_log = []

@fanout_bus.subscribe("events.task")
async def fast_subscriber(msg: Message):
    """Fast subscriber — processes immediately."""
    start = time.time()
    # Simulate fast processing
    await asyncio.sleep(0.01)
    elapsed = time.time() - start
    fanout_log.append({"who": "fast", "msg_id": msg.id[:8], "time": elapsed})
    print(f"  [FastSubscriber]   processed msg {msg.id[:8]} in {elapsed*1000:.0f}ms")

@fanout_bus.subscribe("events.task")
async def slow_subscriber(msg: Message):
    """Slow subscriber — does more work."""
    start = time.time()
    # Simulate slower processing
    await asyncio.sleep(0.05)
    elapsed = time.time() - start
    fanout_log.append({"who": "slow", "msg_id": msg.id[:8], "time": elapsed})
    print(f"  [SlowSubscriber]   processed msg {msg.id[:8]} in {elapsed*1000:.0f}ms")

@fanout_bus.subscribe("events.task")
async def logging_subscriber(msg: Message):
    """Logging subscriber — just records the message."""
    fanout_log.append({"who": "logger", "msg_id": msg.id[:8], "time": 0})
    print(f"  [LoggingSubscriber] recorded msg {msg.id[:8]}")

async def demo_fanout():
    print("Publishing one message — observing fan-out to 3 subscribers")
    print("Subscribers run concurrently; slow subscriber doesn't block fast subscriber.")
    print()
    
    msg = Message(
        topic="events.task",
        content={"task": "process_report", "report_id": "Q4-2025"},
        sender="TaskDispatcher"
    )
    
    start = time.time()
    await fanout_bus.publish(msg)
    await asyncio.sleep(0.1)  # Wait for all subscribers to finish
    total = time.time() - start
    
    print()
    print(f"Total wall time: {total*1000:.0f}ms")
    print("Note: The slow subscriber runs concurrently with fast/logging — no sequential blocking.")

asyncio.run(demo_fanout())

print()
print(f"Fan-out log entries: {len(fanout_log)} (3 subscribers × 1 message)")

## Section 6: Pattern 2 — Request/Reply

Request/Reply implements **synchronous-style remote procedure calls** over the message bus:
- The **requester** sends a message and **awaits a response**
- The **handler** processes the request and sends back a reply
- Correlation IDs ensure the right reply goes to the right requester

This is useful when:
- Agent A needs a computation from Agent B and must wait for the result
- You want request routing without hardcoding which agent handles a request
- Multiple handler agents can serve requests from the same topic (load balancing)

**`@bus.on_request("topic")`** registers a handler. **`await bus.request("topic", content)`** sends a request and waits for the reply.

In [ ]:
# --- Pattern 2: Request/Reply ---

rr_bus = InMemoryBus()

# Register a request handler
@rr_bus.on_request("compute.sentiment")
async def sentiment_handler(msg: Message):
    """
    Handles sentiment analysis requests.
    Receives a Message with text content.
    Must return the response (not publish — the SDK handles reply routing).
    """
    text = msg.content if isinstance(msg.content, str) else str(msg.content)
    print(f"  [SentimentHandler] Received request from {msg.sender}: '{text[:50]}'")
    
    # Simulate sentiment analysis
    import random
    await asyncio.sleep(0.02)  # Simulate processing time
    
    score = random.uniform(-1.0, 1.0)
    label = "positive" if score > 0.1 else ("negative" if score < -0.1 else "neutral")
    
    response = {
        "text": text,
        "sentiment_score": round(score, 3),
        "label": label,
        "handler": "SentimentAnalysisService"
    }
    print(f"  [SentimentHandler] Responding: score={score:.3f}, label={label}")
    return response

@rr_bus.on_request("compute.classify")
async def classification_handler(msg: Message):
    """
    Handles text classification requests.
    Different topic = different handler.
    """
    text = msg.content if isinstance(msg.content, str) else str(msg.content)
    print(f"  [ClassifyHandler] Received request: '{text[:40]}'")
    
    await asyncio.sleep(0.015)
    categories = ["technology", "finance", "health", "politics", "sports"]
    import random
    result = {"category": random.choice(categories), "confidence": round(random.uniform(0.6, 0.99), 2)}
    print(f"  [ClassifyHandler] Category: {result['category']} (conf={result['confidence']})")
    return result

async def demo_request_reply():
    """
    Demonstrate request/reply pattern.
    The caller awaits the response synchronously.
    """
    print("=" * 60)
    print("REQUEST/REPLY PATTERN")
    print("=" * 60)
    
    # Sentiment analysis request
    text1 = "The new product launch exceeded all expectations!"
    print(f"\nRequesting sentiment analysis for: '{text1[:50]}'")
    
    # await bus.request() sends the message and waits for the reply
    # Returns the response content (not a Message object)
    sentiment_result = await rr_bus.request(
        topic="compute.sentiment",
        content=text1,
        sender="AnalysisPipeline",
        timeout=5.0  # Wait up to 5 seconds for reply
    )
    
    print(f"\nSentiment result: {sentiment_result}")
    
    # Classification request
    text2 = "Federal Reserve announces interest rate decision"
    print(f"\nRequesting classification for: '{text2}'")
    
    classify_result = await rr_bus.request(
        topic="compute.classify",
        content=text2,
        sender="AnalysisPipeline"
    )
    
    print(f"\nClassification result: {classify_result}")
    
    # Multiple requests in parallel
    print("\n--- 3 parallel sentiment requests ---")
    texts = [
        "Amazing results this quarter!",
        "The situation is challenging.",
        "No significant changes observed."
    ]
    
    # Fire all requests concurrently
    results = await asyncio.gather(*[
        rr_bus.request("compute.sentiment", text, sender=f"Requester-{i}")
        for i, text in enumerate(texts)
    ])
    
    print("\nParallel results:")
    for text, result in zip(texts, results):
        print(f"  '{text[:40]}' → {result.get('label')} ({result.get('sentiment_score')})")

asyncio.run(demo_request_reply())

print()
print("Key: request() blocks the caller until the handler responds.")
print("Correlation IDs (auto-managed by the SDK) ensure replies go to the right caller.")

## Section 7: Pattern 3 — Broadcast

**Broadcast** sends a message to **all registered agents** on the bus, regardless of their topic subscriptions.

Use cases:
- System-wide notifications ("shutdown imminent", "configuration changed")
- Heartbeat messages that all agents should acknowledge
- Emergency signals that override normal topic routing

Unlike pub/sub (which routes by topic), broadcast reaches everyone.

In [ ]:
# --- Pattern 3: Broadcast ---

broadcast_bus = InMemoryBus()
broadcast_receipts = []

# Register agents on different topics
@broadcast_bus.subscribe("data.raw")
async def data_processor(msg: Message):
    if msg.topic == "system.broadcast":
        broadcast_receipts.append("DataProcessor")
        print(f"  [DataProcessor]    BROADCAST received: {msg.content}")
    else:
        print(f"  [DataProcessor]    Normal message on {msg.topic}: {msg.content}")

@broadcast_bus.subscribe("results.final")
async def results_aggregator(msg: Message):
    if msg.topic == "system.broadcast":
        broadcast_receipts.append("ResultsAggregator")
        print(f"  [ResultsAggregator] BROADCAST received: {msg.content}")
    else:
        print(f"  [ResultsAggregator] Normal message on {msg.topic}: {msg.content}")

@broadcast_bus.subscribe("logs.audit")
async def audit_logger(msg: Message):
    if msg.topic == "system.broadcast":
        broadcast_receipts.append("AuditLogger")
        print(f"  [AuditLogger]      BROADCAST received: {msg.content}")
    else:
        print(f"  [AuditLogger]      Normal message on {msg.topic}: {msg.content}")

async def demo_broadcast():
    print("=" * 60)
    print("BROADCAST PATTERN")
    print("=" * 60)
    print("Agents subscribed to different topics:")
    print("  DataProcessor      → 'data.raw'")
    print("  ResultsAggregator  → 'results.final'")
    print("  AuditLogger        → 'logs.audit'")
    print()
    
    # First send a regular message — only data.raw subscriber receives it
    print("Sending normal message to 'data.raw' topic:")
    await broadcast_bus.publish(Message(topic="data.raw", content="Normal data payload"))
    await asyncio.sleep(0.02)
    
    print()
    print("Now broadcasting to ALL agents (topic-independent):")
    
    # Broadcast: reaches all agents regardless of their subscribed topic
    broadcast_msg = Message(
        topic="system.broadcast",
        content="SYSTEM: Pipeline shutting down in 30 seconds. Please flush buffers.",
        sender="SystemController",
        priority=100  # Highest priority
    )
    
    await broadcast_bus.broadcast(broadcast_msg)
    await asyncio.sleep(0.05)
    
    print()
    print(f"Broadcast receipts: {broadcast_receipts}")
    print(f"All 3 agents received the broadcast? {len(broadcast_receipts) == 3}")

asyncio.run(demo_broadcast())

## Section 8: Pattern 4 — Push/Pull (Work Queue)

**Push/Pull** implements a **work queue** with multiple competing consumers:
- A **producer** pushes work items to a named queue
- Multiple **workers** are registered to that queue
- The bus distributes items **round-robin** across workers (each item goes to exactly ONE worker)

This is different from pub/sub (where ALL subscribers get each message). In push/pull, each message is processed by exactly one worker — which is what you want for distributing computational work.

Use cases:
- Parallel document processing (10 workers, each takes one document)
- LLM request load balancing across multiple API keys or endpoints
- Batch processing pipelines with variable worker availability

In [ ]:
# --- Pattern 4: Push/Pull (Work Queue) ---

workqueue_bus = InMemoryBus()
work_completed = []

# Define worker functions
async def worker_a(content):
    """Worker A: Processes documents."""
    await asyncio.sleep(0.02)  # Simulate processing
    result = f"WorkerA processed: {content}"
    work_completed.append({"worker": "A", "content": content, "result": result})
    print(f"  [Worker A] Processing: {content!r} → Done")
    return result

async def worker_b(content):
    """Worker B: Processes documents."""
    await asyncio.sleep(0.03)  # Slightly slower
    result = f"WorkerB processed: {content}"
    work_completed.append({"worker": "B", "content": content, "result": result})
    print(f"  [Worker B] Processing: {content!r} → Done")
    return result

async def worker_c(content):
    """Worker C: Processes documents."""
    await asyncio.sleep(0.01)  # Fastest
    result = f"WorkerC processed: {content}"
    work_completed.append({"worker": "C", "content": content, "result": result})
    print(f"  [Worker C] Processing: {content!r} → Done")
    return result

# Register workers on the 'documents.process' queue
# All three workers compete for jobs — each job goes to exactly ONE worker (round-robin)
workqueue_bus.add_worker("documents.process", worker_a)
workqueue_bus.add_worker("documents.process", worker_b)
workqueue_bus.add_worker("documents.process", worker_c)

print("Work queue configured: 3 workers on 'documents.process'")
print("Round-robin distribution: Doc1→A, Doc2→B, Doc3→C, Doc4→A, ...")
print()

async def demo_push_pull():
    # Push 9 jobs — should be distributed round-robin across 3 workers
    documents = [f"Document_{i:02d}.pdf" for i in range(1, 10)]
    
    print(f"Pushing {len(documents)} documents to queue:")
    for doc in documents:
        await workqueue_bus.push("documents.process", doc)
    
    # Wait for all to complete
    await asyncio.sleep(0.2)

asyncio.run(demo_push_pull())

print()
print("=" * 60)
print("WORK DISTRIBUTION RESULTS")
print("=" * 60)

from collections import Counter
worker_counts = Counter(item["worker"] for item in work_completed)
print(f"Total work items completed: {len(work_completed)}")
print("Work distribution (round-robin):")
for worker, count in sorted(worker_counts.items()):
    print(f"  Worker {worker}: {count} items ({count/len(work_completed)*100:.0f}%)")

print()
print("Key: Each document was processed by EXACTLY ONE worker.")
print("This is unlike pub/sub where every subscriber gets every message.")

## Section 9: Pattern 5 — Direct Queue (Actor Pattern)

**Direct Queue** implements the **Actor model** — each agent has its own inbox queue, identified by its `agent_id`.

- `await bus.send("agent_id", content)` — puts a message in the named agent's inbox
- `msg = await bus.receive("agent_id")` — blocks until a message arrives in the inbox

This is **point-to-point** messaging — unlike pub/sub (fan-out) or push/pull (any worker).

Use cases:
- Coordination between specific agents (A sends a signal specifically to B)
- Long-running agents that poll their inbox for new tasks
- Stateful agents that need sequenced, ordered message delivery

In [ ]:
# --- Pattern 5: Direct Queue (Actor Pattern) ---

actor_bus = InMemoryBus()

async def coordinator_actor():
    """
    Coordinator actor: sends specific tasks to specific workers.
    Uses direct queue to target a specific worker by ID.
    """
    print("  [Coordinator] Starting — will send 3 tasks to Worker-Alpha")
    
    tasks = [
        {"id": 1, "action": "fetch_data",    "url": "https://api.example.com/data/1"},
        {"id": 2, "action": "process_report", "report": "Q4-annual"},
        {"id": 3, "action": "notify_team",    "channel": "#alerts"}
    ]
    
    for task in tasks:
        # Send directly to Worker-Alpha's inbox
        await actor_bus.send("Worker-Alpha", task)
        print(f"  [Coordinator] Sent task {task['id']} ({task['action']}) to Worker-Alpha")
        await asyncio.sleep(0.01)
    
    # Send a STOP sentinel to tell the worker to stop listening
    await actor_bus.send("Worker-Alpha", {"action": "STOP"})
    print("  [Coordinator] Sent STOP signal to Worker-Alpha")

async def worker_alpha_actor():
    """
    Worker-Alpha actor: polls its own inbox for tasks.
    Only this actor reads from the 'Worker-Alpha' queue.
    """
    print("  [Worker-Alpha] Started — listening to my inbox")
    results = []
    
    while True:
        # Block until a message arrives in this agent's inbox
        msg = await actor_bus.receive("Worker-Alpha", timeout=2.0)
        
        if msg is None:  # Timeout
            print("  [Worker-Alpha] Timeout — no more messages")
            break
        
        content = msg.content
        
        if content.get("action") == "STOP":
            print("  [Worker-Alpha] Received STOP — shutting down")
            break
        
        # Process the task
        await asyncio.sleep(0.02)  # Simulate work
        result = f"Completed {content['action']} (task {content['id']})"
        results.append(result)
        print(f"  [Worker-Alpha] Processed: {result}")
    
    return results

async def demo_direct_queue():
    print("=" * 60)
    print("DIRECT QUEUE / ACTOR PATTERN")
    print("=" * 60)
    print()
    
    # Run coordinator and worker concurrently
    coord_task = asyncio.create_task(coordinator_actor())
    worker_task = asyncio.create_task(worker_alpha_actor())
    
    await asyncio.gather(coord_task, worker_task)

asyncio.run(demo_direct_queue())

print()
print("Key: Only Worker-Alpha could read from the 'Worker-Alpha' queue.")
print("Other agents sending to different IDs would have separate inboxes.")

## Section 10: TTL Expiry

Messages with `ttl=N` expire after N seconds. If a subscriber doesn't receive the message within TTL, it is discarded and moved to the **dead-letter queue**.

This prevents stale work from piling up when consumers are slow or temporarily unavailable.

In [ ]:
# --- TTL expiry demo ---

ttl_bus = InMemoryBus()
received_before_expiry = []

@ttl_bus.subscribe("time.sensitive")
async def time_sensitive_handler(msg: Message):
    """Handles time-sensitive messages. Will only receive non-expired messages."""
    received_before_expiry.append(msg.id)
    print(f"  [Handler] Received (within TTL): id={msg.id[:8]}, content={msg.content!r}")

async def demo_ttl():
    print("=" * 60)
    print("TTL EXPIRY DEMO")
    print("=" * 60)
    
    print("\nPublishing message with TTL=5 seconds (will NOT expire quickly):")
    alive_msg = Message(
        topic="time.sensitive",
        content="Urgent: process immediately",
        ttl=5.0,  # Will live for 5 seconds
        sender="AlertSystem"
    )
    await ttl_bus.publish(alive_msg)
    await asyncio.sleep(0.05)  # Process quickly, well within TTL
    print(f"  Delivered: {alive_msg.id[:8]}")
    
    print("\nPublishing message with TTL=0.001 seconds (will expire before delivery):")
    expired_msg = Message(
        topic="time.sensitive",
        content="This message will expire",
        ttl=0.001,  # Expires in 1ms — will be dead on arrival
        sender="AlertSystem"
    )
    await ttl_bus.publish(expired_msg)
    await asyncio.sleep(0.1)  # Wait longer than TTL
    
    print()
    print(f"Messages received before TTL: {len(received_before_expiry)}")
    print(f"IDs received: {[mid[:8] for mid in received_before_expiry]}")
    print()
    print("The expired message (TTL=0.001s) was NOT delivered — it's in the dead-letter queue.")

asyncio.run(demo_ttl())

## Section 11: Message Priority

When multiple messages are queued, **higher priority messages are processed first**.

`Message(priority=N)` — N can be any integer. Higher N = processed first.

Use cases:
- Error alerts should jump ahead of normal processing messages
- User-facing requests should be prioritized over background batch work
- Critical system signals should be processed before low-priority analytics

In [ ]:
# --- Message priority demo ---

priority_bus = InMemoryBus()
processing_order = []

@priority_bus.subscribe("tasks.queue")
async def priority_handler(msg: Message):
    """Processes messages in priority order (highest first)."""
    processing_order.append((msg.priority, msg.content))
    print(f"  [Handler] Processing priority={msg.priority:3d}: {msg.content!r}")

async def demo_priority():
    print("=" * 60)
    print("MESSAGE PRIORITY DEMO")
    print("=" * 60)
    print("Publishing messages with different priorities...")
    print("Messages are queued, then processed highest-priority-first.")
    print()
    
    messages = [
        Message(topic="tasks.queue", content="Background analytics job",    priority=1),
        Message(topic="tasks.queue", content="Normal user request",         priority=5),
        Message(topic="tasks.queue", content="CRITICAL: System alert",      priority=100),
        Message(topic="tasks.queue", content="Batch export job",            priority=2),
        Message(topic="tasks.queue", content="High-priority user request",  priority=50),
        Message(topic="tasks.queue", content="Routine health check",        priority=0),
        Message(topic="tasks.queue", content="URGENT: Payment processing", priority=90),
    ]
    
    print("Publishing in this order (NOT priority order):")
    for msg in messages:
        print(f"  priority={msg.priority:3d}: {msg.content}")
        await priority_bus.publish(msg)
    
    await asyncio.sleep(0.1)  # Allow processing
    
    print()
    print("Actual processing order (highest priority first):")
    for i, (prio, content) in enumerate(processing_order, 1):
        print(f"  {i}. priority={prio:3d}: {content}")

asyncio.run(demo_priority())

print()
print("Observation: CRITICAL (100), URGENT (90), High-priority (50) processed first,")
print("even though they were published in the middle of the list.")

## Section 12: Dead-Letter Queue

When a message **fails to deliver** (subscriber throws an exception) or **expires** (TTL exceeded), it goes to the **dead-letter queue (DLQ)**.

The DLQ allows you to:
- Inspect failed messages for debugging
- Replay messages after fixing the subscriber issue
- Alert on unexpected failure rates

In [ ]:
# --- Dead-letter queue demo ---

dlq_bus = InMemoryBus()

@dlq_bus.subscribe("risky.operations")
async def failing_subscriber(msg: Message):
    """This subscriber intentionally raises an exception for certain messages."""
    content = msg.content
    if isinstance(content, dict) and content.get("should_fail"):
        raise ValueError(f"Simulated processing failure for message: {msg.id[:8]}")
    print(f"  [RiskyHandler] Successfully processed: {content}")

async def demo_dlq():
    print("=" * 60)
    print("DEAD-LETTER QUEUE DEMO")
    print("=" * 60)
    print()
    
    # Publish messages — some will fail, some will succeed
    messages = [
        Message(topic="risky.operations", content={"task": "report_gen", "should_fail": False}),
        Message(topic="risky.operations", content={"task": "data_import", "should_fail": True}),
        Message(topic="risky.operations", content={"task": "email_send",  "should_fail": False}),
        Message(topic="risky.operations", content={"task": "api_call",    "should_fail": True}),
        Message(topic="risky.operations", content={"task": "log_write",   "should_fail": False}),
    ]
    
    for msg in messages:
        await dlq_bus.publish(msg)
    
    await asyncio.sleep(0.1)
    
    # Inspect the dead-letter queue
    dlq_snapshot = dlq_bus.dlq_snapshot()
    
    print(f"Dead-letter queue contents ({len(dlq_snapshot)} failed messages):")
    for dead_msg in dlq_snapshot:
        print(f"  id={dead_msg.id[:8]}, content={dead_msg.content}, reason=subscriber exception")
    
    print()
    print("Flushing (replaying) the DLQ after 'fixing' the subscriber:")
    
    # In production, you'd fix the subscriber first, then flush
    # dlq_flush() requeues all DLQ messages for redelivery
    replayed = dlq_bus.dlq_flush()
    print(f"  {replayed} messages requeued for retry")

asyncio.run(demo_dlq())

## Section 13: Bus Stats

The `bus.stats` property returns a snapshot of messaging activity — useful for monitoring, debugging, and performance tuning.

In [ ]:
# --- Bus statistics ---

stats_bus = InMemoryBus()

@stats_bus.subscribe("work.tasks")
async def stats_worker(msg: Message):
    # Randomly fail some messages for demo
    import random
    if random.random() < 0.2:
        raise RuntimeError("Random processing failure")
    await asyncio.sleep(0.005)

async def demo_stats():
    # Publish 20 messages
    for i in range(20):
        await stats_bus.publish(Message(
            topic="work.tasks",
            content=f"task_{i:03d}",
            sender="Producer"
        ))
    
    await asyncio.sleep(0.3)  # Let messages process
    
    # Access bus statistics
    stats = stats_bus.stats
    
    print("=" * 60)
    print("BUS STATISTICS")
    print("=" * 60)
    print(f"published:          {stats['published']}     ← total messages published")
    print(f"delivered:          {stats['delivered']}     ← successfully delivered")
    print(f"failed:             {stats['failed']}      ← delivery failures")
    print(f"dead_letters:       {stats['dead_letters']}      ← in DLQ")
    print(f"subscribers_active: {stats['subscribers_active']}      ← registered subscribers")
    print(f"queues_active:      {stats['queues_active']}      ← named queues")
    print()
    
    if stats['published'] > 0:
        success_rate = stats['delivered'] / stats['published'] * 100
        print(f"Delivery success rate: {success_rate:.1f}%")

asyncio.run(demo_stats())

## Section 14: Integrating Bus with Chain

A `Chain` can publish messages to the bus during execution — allowing it to notify other parts of the system about its progress without tight coupling.

This is the **event-driven architecture** pattern: agents emit events, and interested consumers react.

In [ ]:
# --- Integrating the bus with a Chain ---

from multigen import Chain

integration_bus = InMemoryBus()
pipeline_events = []

# Subscribe to pipeline lifecycle events
@integration_bus.subscribe("pipeline.events")
async def pipeline_monitor(msg: Message):
    """Monitors all pipeline events — acts as an observability sink."""
    pipeline_events.append(msg.content)
    print(f"  [Monitor] Event: {msg.content.get('event', 'unknown')} | stage: {msg.content.get('stage')} | sender: {msg.sender}")

@integration_bus.subscribe("pipeline.results")
async def results_handler(msg: Message):
    """Receives final results for downstream processing."""
    print(f"  [ResultsHandler] Final output received from {msg.sender}")

# Agents that publish to the bus as they execute
async def bus_aware_extract(query, context=None):
    """Agent that publishes its start/complete events to the bus."""
    # Publish start event
    await integration_bus.publish(Message(
        topic="pipeline.events",
        content={"event": "started", "stage": "extract", "input": str(query)[:50]},
        sender="ExtractAgent"
    ))
    
    await asyncio.sleep(0.02)
    result = {"data": f"Extracted from: {query}", "records": 42}
    
    # Publish complete event
    await integration_bus.publish(Message(
        topic="pipeline.events",
        content={"event": "completed", "stage": "extract", "records": 42},
        sender="ExtractAgent"
    ))
    return result

async def bus_aware_transform(data, context=None):
    """Transform agent with bus integration."""
    await integration_bus.publish(Message(
        topic="pipeline.events",
        content={"event": "started", "stage": "transform"},
        sender="TransformAgent"
    ))
    
    await asyncio.sleep(0.015)
    records = data.get("records", 0) if isinstance(data, dict) else 0
    result = {"transformed_records": records, "format": "normalized"}
    
    await integration_bus.publish(Message(
        topic="pipeline.events",
        content={"event": "completed", "stage": "transform", "output_records": records},
        sender="TransformAgent"
    ))
    return result

async def bus_aware_load(data, context=None):
    """Load agent — publishes result to a separate topic too."""
    await integration_bus.publish(Message(
        topic="pipeline.events",
        content={"event": "started", "stage": "load"},
        sender="LoadAgent"
    ))
    
    await asyncio.sleep(0.01)
    result = {"loaded": True, "destination": "data_warehouse", "records": data.get("transformed_records", 0)}
    
    await integration_bus.publish(Message(
        topic="pipeline.events",
        content={"event": "completed", "stage": "load", "success": True},
        sender="LoadAgent"
    ))
    
    # Publish final result to results topic
    await integration_bus.publish(Message(
        topic="pipeline.results",
        content=result,
        sender="LoadAgent"
    ))
    return result

# Build the chain
etl_chain = Chain(
    FunctionAgent(bus_aware_extract,   name="Extract"),
    FunctionAgent(bus_aware_transform, name="Transform"),
    FunctionAgent(bus_aware_load,      name="Load"),
    name="ETL_Pipeline"
)

async def demo_bus_chain():
    print("=" * 60)
    print("CHAIN + BUS INTEGRATION")
    print("=" * 60)
    print("Running ETL chain — agents publish events to bus as they execute")
    print()
    
    result = await etl_chain.run("SELECT * FROM transactions WHERE date > 2025-01-01")
    await asyncio.sleep(0.1)  # Let all events propagate
    
    print()
    print(f"Chain result: {result}")
    print(f"Total pipeline events published: {len(pipeline_events)}")

asyncio.run(demo_bus_chain())

## Section 15: Real-World Example — Multi-Agent Pub/Sub Pipeline

Let's build a realistic multi-agent news analysis pipeline:

```
NewsIngestionAgent
       │
       │ publishes to 'news.articles'
       ▼
  [news.articles topic]
       │
       ├──► SentimentAnalysisAgent   (subscriber 1)
       ├──► EntityExtractionAgent    (subscriber 2)
       └──► TopicClassifierAgent     (subscriber 3)
```

The producer agent doesn't know about the 3 consumers. Each consumer does independent analysis. Results are published to their own topics for downstream aggregation.

In [ ]:
# --- Real-world multi-agent pub/sub pipeline ---

import random

pipeline_bus = InMemoryBus()
analysis_results = {"sentiment": [], "entities": [], "topics": []}

# --- Consumer 1: Sentiment Analysis Agent ---
@pipeline_bus.subscribe("news.articles")
async def sentiment_analysis_agent(msg: Message):
    """Analyzes the emotional tone of each article."""
    article = msg.content
    text = article.get("title", "") + " " + article.get("body", "")
    
    # Simulate LLM sentiment call
    await asyncio.sleep(random.uniform(0.01, 0.05))
    score = random.uniform(-1.0, 1.0)
    label = "positive" if score > 0.2 else ("negative" if score < -0.2 else "neutral")
    
    result = {
        "article_id": article.get("id"),
        "sentiment_score": round(score, 3),
        "label": label,
        "analyzer": "SentimentAgent"
    }
    analysis_results["sentiment"].append(result)
    
    # Publish sentiment result for downstream aggregation
    await pipeline_bus.publish(Message(
        topic="analysis.sentiment",
        content=result,
        sender="SentimentAnalysisAgent"
    ))

# --- Consumer 2: Entity Extraction Agent ---
@pipeline_bus.subscribe("news.articles")
async def entity_extraction_agent(msg: Message):
    """Extracts named entities (people, organizations, locations)."""
    article = msg.content
    
    await asyncio.sleep(random.uniform(0.02, 0.06))
    
    # Simulate NER
    sample_people = ["Elon Musk", "Janet Yellen", "Tim Cook", "Sam Altman"]
    sample_orgs   = ["Apple", "Tesla", "Federal Reserve", "OpenAI", "Google"]
    sample_places = ["New York", "Silicon Valley", "Washington D.C.", "London"]
    
    result = {
        "article_id": article.get("id"),
        "people":   random.sample(sample_people, k=random.randint(0, 2)),
        "orgs":     random.sample(sample_orgs,   k=random.randint(1, 3)),
        "places":   random.sample(sample_places, k=random.randint(0, 2)),
        "analyzer": "EntityExtractionAgent"
    }
    analysis_results["entities"].append(result)
    
    await pipeline_bus.publish(Message(
        topic="analysis.entities",
        content=result,
        sender="EntityExtractionAgent"
    ))

# --- Consumer 3: Topic Classifier Agent ---
@pipeline_bus.subscribe("news.articles")
async def topic_classifier_agent(msg: Message):
    """Classifies each article into one of 5 news categories."""
    article = msg.content
    
    await asyncio.sleep(random.uniform(0.01, 0.04))
    
    categories = ["technology", "finance", "politics", "health", "environment"]
    primary = random.choice(categories)
    secondary = random.choice([c for c in categories if c != primary])
    
    result = {
        "article_id": article.get("id"),
        "primary_category": primary,
        "secondary_category": secondary,
        "confidence": round(random.uniform(0.65, 0.99), 2),
        "analyzer": "TopicClassifierAgent"
    }
    analysis_results["topics"].append(result)
    
    await pipeline_bus.publish(Message(
        topic="analysis.topics",
        content=result,
        sender="TopicClassifierAgent"
    ))

# --- Aggregator: Listens to all three analysis topics ---
aggregated = []

@pipeline_bus.subscribe("analysis.sentiment")
@pipeline_bus.subscribe("analysis.entities")
@pipeline_bus.subscribe("analysis.topics")
async def aggregator_agent(msg: Message):
    """Aggregates results from all three analysis agents."""
    aggregated.append({"topic": msg.topic, "data": msg.content})

# --- Producer: News Ingestion Agent ---
async def news_ingestion_agent(articles):
    """Ingests articles and publishes them to the news.articles topic."""
    for article in articles:
        await pipeline_bus.publish(Message(
            topic="news.articles",
            content=article,
            sender="NewsIngestionAgent",
            headers={"ingestion_ts": str(time.time())}
        ))
        await asyncio.sleep(0.005)  # Small delay between articles

async def run_full_pipeline():
    # Sample news articles
    articles = [
        {"id": "art-001", "title": "Apple announces new AI chip",        "body": "Apple's M4 chip..."},
        {"id": "art-002", "title": "Federal Reserve raises rates",         "body": "The Fed announced..."},
        {"id": "art-003", "title": "Climate accord signed in Geneva",      "body": "World leaders..."},
        {"id": "art-004", "title": "Tesla reports record deliveries",      "body": "Electric vehicle..."},
        {"id": "art-005", "title": "OpenAI launches GPT-5",               "body": "The new model..."},
    ]
    
    print("=" * 70)
    print("MULTI-AGENT PUB/SUB PIPELINE")
    print("=" * 70)
    print(f"Producer: 1 × NewsIngestionAgent")
    print(f"Consumers: 3 × (SentimentAgent, EntityAgent, TopicClassifierAgent)")
    print(f"Aggregator: 1 × (subscribes to all 3 analysis topics)")
    print(f"Input: {len(articles)} articles")
    print()
    print("Running pipeline...")
    
    await news_ingestion_agent(articles)
    await asyncio.sleep(0.5)  # Wait for all async processing
    
    # Summary
    print()
    print("=" * 70)
    print("PIPELINE RESULTS")
    print("=" * 70)
    
    print(f"\nSentiment results ({len(analysis_results['sentiment'])}/{len(articles)} articles):")
    for r in analysis_results["sentiment"]:
        print(f"  {r['article_id']}: {r['label']} (score={r['sentiment_score']})")
    
    print(f"\nEntity extraction results ({len(analysis_results['entities'])}/{len(articles)} articles):")
    for r in analysis_results["entities"]:
        print(f"  {r['article_id']}: orgs={r['orgs']}, people={r['people']}")
    
    print(f"\nTopic classification ({len(analysis_results['topics'])}/{len(articles)} articles):")
    for r in analysis_results["topics"]:
        print(f"  {r['article_id']}: {r['primary_category']} / {r['secondary_category']} (conf={r['confidence']})")
    
    print(f"\nAggregated messages received: {len(aggregated)}")
    print(f"Expected: {len(articles)} articles × 3 analysis types = {len(articles)*3}")
    
    # Bus stats
    stats = pipeline_bus.stats
    print(f"\nBus stats: published={stats['published']}, delivered={stats['delivered']}, failed={stats['failed']}")

asyncio.run(run_full_pipeline())

## Summary and Key Takeaways

In this notebook we explored the full range of messaging patterns available via `InMemoryBus`:

### Pattern Summary

| Pattern | Method | Use Case |
|---|---|---|
| **Pub/Sub** | `@bus.subscribe`, `bus.publish` | Fan-out: one producer, many consumers |
| **Request/Reply** | `@bus.on_request`, `bus.request` | Synchronous-style RPC over messaging |
| **Broadcast** | `bus.broadcast` | System-wide signals to all agents |
| **Push/Pull** | `bus.add_worker`, `bus.push` | Load-balanced work queue |
| **Direct Queue** | `bus.send`, `bus.receive` | Actor model point-to-point |

### When to Use Each Pattern

- **Pub/Sub**: Any time you want **decoupled fan-out** — producer doesn't know consumers
- **Request/Reply**: When you need a **response** to a specific request (like HTTP GET)
- **Broadcast**: System events that **everyone must know about** (shutdown, config change)
- **Push/Pull**: **Parallel work distribution** across multiple workers
- **Direct Queue**: **Stateful agents** with private inboxes, Actor model

### Next Steps

- **Notebook 18**: Runtime — Unified Execution with Observability and Backends
- **Notebook 19**: Resilience — Circuit Breakers, Retry, Memory, Human-in-the-Loop